In [2]:
import numpy as np
import pandas as pd

In [3]:
df1 = pd.read_csv('covid_19_india.csv')
df2 = pd.read_csv('StatewiseTestingDetails.csv')

In [4]:
df1.head(10)
df2.head(10)

,Date,State,TotalSamples,Negative,Positive
0,2020-04-17,Andaman and Nicobar Islands,1403.0,1210,12.0
1,2020-04-24,Andaman and Nicobar Islands,2679.0,NaN,27.0
2,2020-04-27,Andaman and Nicobar Islands,2848.0,NaN,33.0
3,2020-05-01,Andaman and Nicobar Islands,3754.0,NaN,33.0
4,2020-05-16,Andaman and Nicobar Islands,6677.0,NaN,33.0
5,2020-05-19,Andaman and Nicobar Islands,6965.0,NaN,33.0
6,2020-05-20,Andaman and Nicobar Islands,7082.0,NaN,33.0
7,2020-05-21,Andaman and Nicobar Islands,7167.0,NaN,33.0
8,2020-05-22,Andaman and Nicobar Islands,7263.0,NaN,33.0
9,2020-05-23,Andaman and Nicobar Islands,7327.0,NaN,33.0


In [5]:
df1.info()
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 18110 entries, 0 to 18109
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Sno                       18110 non-null  int64
 1   Date                      18110 non-null  str  
 2   Time                      18110 non-null  str  
 3   State/UnionTerritory      18110 non-null  str  
 4   ConfirmedIndianNational   18110 non-null  str  
 5   ConfirmedForeignNational  18110 non-null  str  
 6   Cured                     18110 non-null  int64
 7   Deaths                    18110 non-null  int64
 8   Confirmed                 18110 non-null  int64
dtypes: int64(4), str(5)
memory usage: 1.2 MB
<class 'pandas.DataFrame'>
RangeIndex: 16336 entries, 0 to 16335
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          16336 non-null  str    
 1   State         16336 non-null  str    
 2   

In [6]:
df1.rename(columns={"State/UnionTerritory": "State"}, inplace=True)
df1.head()

,Sno,Date,Time,State,ConfirmedIndianNational,ConfirmedForeignNational,Cured,Deaths,Confirmed
0,1,2020-01-30,6:00 PM,Kerala,1,0,0,0,1
1,2,2020-01-31,6:00 PM,Kerala,1,0,0,0,1
2,3,2020-02-01,6:00 PM,Kerala,2,0,0,0,2
3,4,2020-02-02,6:00 PM,Kerala,3,0,0,0,3
4,5,2020-02-03,6:00 PM,Kerala,3,0,0,0,3


In [7]:
duplicate_list = pd.merge(df1, df2)
unique_list= duplicate_list['State'].unique().tolist()
len(unique_list)

36

In [9]:
df1.isnull().sum()
df2.isnull().sum()


Date                0
State               0
TotalSamples        0
Negative         9367
Positive        10674
dtype: int64

In [11]:
df2 = df2.fillna(0)
df2.isnull().sum()

Date            0
State           0
TotalSamples    0
Negative        0
Positive        0
dtype: int64

In [12]:
data1 = df1[df1['State'].isin(unique_list) & (df1['Date'] >= '2020-04-17') & (df1['Date'] <= '2021-08-10')]

In [26]:
# Use the states that exist in both datasets and the same date window
data2 = (
    df2.loc[
        df2["State"].isin(unique_list) &
        (df2["Date"] >= "2020-04-17") &
        (df2["Date"] <= "2021-08-10"),
        ["Date", "State", "TotalSamples", "Positive"]
    ]
    .copy()
)

# Clean numeric columns
data2["TotalSamples"] = pd.to_numeric(data2["TotalSamples"], errors="coerce")
data2["Positive"] = pd.to_numeric(data2["Positive"], errors="coerce")

# Drop rows with missing values
data2 = data2.dropna(subset=["TotalSamples", "Positive"]).reset_index(drop=True)

# Optional: add positive rate
data2["Positive Rate"] = data2["Positive"] / data2["TotalSamples"].replace(0, pd.NA)

data2.head()

,Date,State,TotalSamples,Positive,Positive Rate
0,2020-04-17,Andaman and Nicobar Islands,1403.0,12.0,0.008553
1,2020-04-24,Andaman and Nicobar Islands,2679.0,27.0,0.010078
2,2020-04-27,Andaman and Nicobar Islands,2848.0,33.0,0.011587
3,2020-05-01,Andaman and Nicobar Islands,3754.0,33.0,0.008791
4,2020-05-16,Andaman and Nicobar Islands,6677.0,33.0,0.004942


In [28]:
data1 = data1.groupby('State')[['Sno', 'Cured', 'Deaths', 'Confirmed']].sum()
data1

,Sno,Cured,Deaths,Confirmed
State,,,,
Andaman and Nicobar Islands,4547284,1840824,27007,1930746
Andhra Pradesh,4547501,368473679,2925728,390442544
Arunachal Pradesh,4548114,6540328,26551,7126288
Assam,4548595,92118991,632897,99260468
Bihar,4517346,124407421,1083794,131505079
Chandigarh,4549557,10055815,146883,10796259
Chhattisgarh,4550038,150621049,2050376,162772593
Dadra and Nagar Haveli and Daman and Diu,4494036,1831104,1010,1927978
Delhi,4550982,272008186,4917990,285777762


In [29]:
data2 = data2.groupby('State')[['TotalSamples', 'Positive']].sum()
data2

,TotalSamples,Positive
State,,
Andaman and Nicobar Islands,8.747008e+07,1763591.0
Andhra Pradesh,4.967700e+09,3856013.0
Arunachal Pradesh,1.636091e+08,51243.0
Assam,2.853488e+09,2065791.0
Bihar,7.392733e+09,1858782.0
Chandigarh,9.974472e+07,59016.0
Chhattisgarh,1.863097e+09,467651.0
Dadra and Nagar Haveli and Daman and Diu,6.323108e+06,169010.0
Delhi,4.310471e+09,6837462.0


In [19]:
data2['Positive Rate'] = data2['Positive'] / data2['TotalSamples']

In [30]:
pd.merge(data1, data2, on='State')

,Sno,Cured,Deaths,Confirmed,TotalSamples,Positive
State,,,,,,
Andaman and Nicobar Islands,4547284,1840824,27007,1930746,8.747008e+07,1763591.0
Andhra Pradesh,4547501,368473679,2925728,390442544,4.967700e+09,3856013.0
Arunachal Pradesh,4548114,6540328,26551,7126288,1.636091e+08,51243.0
Assam,4548595,92118991,632897,99260468,2.853488e+09,2065791.0
Bihar,4517346,124407421,1083794,131505079,7.392733e+09,1858782.0
Chandigarh,4549557,10055815,146883,10796259,9.974472e+07,59016.0
Chhattisgarh,4550038,150621049,2050376,162772593,1.863097e+09,467651.0
Dadra and Nagar Haveli and Daman and Diu,4494036,1831104,1010,1927978,6.323108e+06,169010.0
Delhi,4550982,272008186,4917990,285777762,4.310471e+09,6837462.0
